# AIC26 Preprocessing Pipeline — Kaggle Runner

**Setup:** GPU T4/P100 · Internet ON · Attach video dataset

**Repository:** `https://github.com/Hoaiduc195/aic2026.git`
**Subpackage:** `pipelines/preprocessing`

Edit `GIT_REPO_URL` and `VIDEO_GLOB` below, then Run All.

In [ ]:
# --- CONFIGURATION ---
USE_GITHUB = True  # Set to True to clone from GitHub, or False to use Kaggle Dataset
GIT_REPO_URL = "https://github.com/Hoaiduc195/aic2026.git"  # <--- GitHub Team Repository
GIT_BRANCH = "main"

# Fallback code dataset (if USE_GITHUB = False)
CODE_DS = "/kaggle/input/datasets/khoahope/aic26-pipeline-code"

VIDEO_GLOB = "/kaggle/input/datasets/aresusayhi/ai-challenge-2025/Videos/Videos/**/*.mp4"
OUT = "/kaggle/working/outputs"
LIMIT_HOURS = 35  # stratified subset spanning ~N hours across all batches (L21..L30)
                  # set to None to process the entire dataset (needs multi-commit resume)

# --- CODE SETUP ---
import shutil, os

REPO_DIR = "/kaggle/working/aic2026"
shutil.rmtree(REPO_DIR, ignore_errors=True)  # always fetch fresh code

if USE_GITHUB:
    print(f"Cloning fresh code from GitHub: {GIT_REPO_URL} (branch: {GIT_BRANCH})")
    !git clone --depth 1 -b {GIT_BRANCH} {GIT_REPO_URL} {REPO_DIR}
    # Navigate to the preprocessing pipeline directory
    PIPE_DIR = os.path.join(REPO_DIR, "pipelines", "preprocessing")
    if not os.path.exists(PIPE_DIR):
        PIPE_DIR = os.path.join(REPO_DIR, "aic26-pipeline")  # fallback if nested inside aic26-pipeline
else:
    print(f"Copying code from Kaggle Dataset: {CODE_DS}")
    PIPE_DIR = "/kaggle/working/pipe"
    shutil.copytree(CODE_DS, PIPE_DIR)

%cd {REPO_DIR}
%pip install -q -r {PIPE_DIR}/requirements-kaggle.txt


## (Optional) Resume a previous run
Attach the previous version's output as a dataset and copy it back before running.

In [ ]:
# PREV = "/kaggle/input/previous-run-output/outputs"
# if os.path.exists(PREV) and not os.path.exists(OUT):
#     shutil.copytree(PREV, OUT)

In [ ]:
hours_flag = f"--limit-hours {LIMIT_HOURS}" if LIMIT_HOURS else ""
!python -m pipelines.preprocessing.cli probe     --input-glob "{VIDEO_GLOB}" --out {OUT}
!python -m pipelines.preprocessing.cli shots     --input-glob "{VIDEO_GLOB}" --out {OUT} {hours_flag}
!python -m pipelines.preprocessing.cli extract   --input-glob "{VIDEO_GLOB}" --out {OUT} {hours_flag}
!python -m pipelines.preprocessing.cli index     --out {OUT}
!python -m pipelines.preprocessing.cli benchmark --out {OUT} -k 10 --rounds 5
!python -m pipelines.preprocessing.cli report    --out {OUT}

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f"{OUT}/REPORT.md").read()))

## Validate against organizer (BTC) ground-truth keyframes

Downloads only the BTC `map-keyframes` CSVs for the videos we actually processed
(small files, fast), then checks temporal coverage: for every keyframe BTC's own
system selected, is there one of ours within a time tolerance? This is the
objective "keyframe quality" evidence — not a claim, a measured percentage.

In [ ]:
import os, shutil, glob

# Derive the BTC source from the SAME mount base the videos came from, so this
# cell can't drift out of sync with a hand-edited VIDEO_GLOB in cell-1. Try a
# few known layouts, then fall back to a recursive search.
DATA_ROOT = VIDEO_GLOB.split("/Videos/")[0]
processed = [os.path.splitext(os.path.basename(p))[0]
             for p in glob.glob(f"{OUT}/map-keyframes/*.csv")]
sample = processed[0] if processed else None

CANDIDATES = [
    f"{DATA_ROOT}/features/map-keyframes",
    f"{DATA_ROOT}/map-keyframes",
    f"{DATA_ROOT}/features/map-keyframes/map-keyframes",
    "/kaggle/input/datasets/aresusayhi/ai-challenge-2025/features/map-keyframes",
]
BTC_SRC = None
for c in CANDIDATES:
    if os.path.isdir(c) and glob.glob(f"{c}/*.csv"):
        BTC_SRC = c
        break
if BTC_SRC is None and sample:  # last resort: recursive search for the sample file
    print("candidate paths empty; searching recursively for map-keyframes ...")
    hits = glob.glob(f"{DATA_ROOT}/**/map-keyframes/{sample}.csv", recursive=True)
    if hits:
        BTC_SRC = os.path.dirname(hits[0])

print("DATA_ROOT :", DATA_ROOT)
print("BTC_SRC   :", BTC_SRC)

BTC_DIR = "/kaggle/working/btc_keyframes"
os.makedirs(BTC_DIR, exist_ok=True)
matched = 0
if BTC_SRC:
    for vid in processed:
        # BTC files are keyed by original stem; our video_id may carry a
        # 'video_' disambiguation prefix (see aicpp/eval_btc.py::_btc_path_for)
        for cand in (vid, vid.removeprefix("video_")):
            src = f"{BTC_SRC}/{cand}.csv"
            if os.path.exists(src):
                shutil.copy(src, f"{BTC_DIR}/{vid}.csv")
                matched += 1
                break

print(f"BTC ground truth found for {matched}/{len(processed)} processed videos")
if matched == 0:  # show what actually exists so the cause is obvious, not silent
    for root in (f"{DATA_ROOT}/features", DATA_ROOT):
        if os.path.isdir(root):
            print(f"contents of {root}: {sorted(os.listdir(root))[:25]}")
            break

# eval-btc now always writes EVAL_BTC.md (a diagnostic one if matched==0)
!python -m aicpp.cli eval-btc --out {OUT} --btc-dir {BTC_DIR}

In [ ]:
import os
_p = f"{OUT}/EVAL_BTC.md"
display(Markdown(open(_p).read()) if os.path.exists(_p)
        else Markdown("_EVAL_BTC.md not found — the BTC eval cell above did not complete._"))

In [ ]:
# Bundle just the RESULT files into one small zip for easy download.
# Excludes: the copied source code (/kaggle/working/pipe), keyframe images, and
# feature vectors (.npy) + the FAISS binary -- i.e. everything heavy. What's left
# is the numbers: reports, per-video map-keyframes/metadata/shots, index metadata.
import zipfile, os

RESULTS_ZIP = "/kaggle/working/aic26_results.zip"
INCLUDE_FILES = ["REPORT.md", "EVAL_BTC.md", "eval_btc_per_video.csv",
                 "env.json", "videos_manifest.parquet", "failed_videos.log"]
INCLUDE_DIRS = ["map-keyframes", "metadata", "shots"]  # small per-video CSV/JSON/parquet

def _keep(path):
    # inside index/, keep only the tiny JSON metadata, not the FAISS binary / table
    return path.endswith(".json")

with zipfile.ZipFile(RESULTS_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in INCLUDE_FILES:
        p = os.path.join(OUT, fn)
        if os.path.exists(p):
            z.write(p, fn)
    for d in INCLUDE_DIRS:
        for dp, _, files in os.walk(os.path.join(OUT, d)):
            for f in files:
                full = os.path.join(dp, f)
                z.write(full, os.path.relpath(full, OUT))
    for dp, _, files in os.walk(os.path.join(OUT, "index")):
        for f in files:
            full = os.path.join(dp, f)
            if _keep(full):
                z.write(full, os.path.relpath(full, OUT))

n = len(zipfile.ZipFile(RESULTS_ZIP).namelist())
size_mb = os.path.getsize(RESULTS_ZIP) / 2**20
print(f"wrote {RESULTS_ZIP}  ({size_mb:.2f} MB, {n} files)")
print("Download just this one file from the committed output -> Output tab -> aic26_results.zip")
print("(source code, keyframe images and feature vectors are NOT included)")

## Search demo — text query → top-k keyframes

In [ ]:
import sys; sys.path.insert(0, "/kaggle/working/pipe")
from aicpp.config import PipelineConfig
from aicpp.store import OutputStore
from aicpp.indexer import search
from aicpp.embed import ClipEmbedder

cfg = PipelineConfig(out_dir=OUT)
store = OutputStore(OUT)
embedder = ClipEmbedder(cfg)  # load once, reuse across queries

QUERY = "a news anchor in a television studio"
hits = search(cfg, store, QUERY, k=8, embedder=embedder)
hits

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, row in zip(axes.flat, hits.itertuples()):
    ax.imshow(Image.open(f"{OUT}/{row.path}"))
    ax.set_title(f"{row.video_id} @ {row.pts_time:.1f}s\nscore {row.score:.3f}", fontsize=9)
    ax.axis("off")
plt.suptitle(QUERY)
plt.tight_layout(); plt.show()